In [ ]:
import pandas as pd

df = pd.read_csv('../windows_ALL2_dac_EPS1_3600_10_unx.csv', index_col=0)

cols = [
    'idxwindow', 'idxdaywindow', 'day', 'dac_family', 'mac', 'seconds',
    'qr', 'num DAC_RANK=1', 'num DAC_RANK=3', 'q', 'nx', 'qr_p1', 'q_p1',
    'nx_p1', 'mac2', 'num DN', 'avg DN/QR', 'avg DN/Q', 'avg DN/NX',
    'std DN/QR', 'std DN/Q', 'std DN/NX', 'max DN/QR', 'max DN/Q',
    'max DN/NX'
]

df[(df.idxwindow == 2) & ((df.dac_family == 'virut') | (df.dac_family == 'tofsee'))].sort_values(by=['idxwindow', 'mac'])

macs = df.mac.drop_duplicates().to_numpy().tolist()
dac_families = df.dac_family.drop_duplicates().to_numpy().tolist()


In [2]:
import numpy as np

df[['qr', 'num DAC_RANK=1', 'num DAC_RANK=3', 'q', 'nx', 'qr_p1', 'q_p1',
    'nx_p1', 'mac2', 'num DN', 'avg DN/QR', 'avg DN/Q', 'avg DN/NX',
    'std DN/QR', 'std DN/Q', 'std DN/NX', 'max DN/QR', 'max DN/Q',
    'max DN/NX'
]].to_numpy().tolist()[0]

emptyrow = [0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 'nx',
 0,
 np.nan,
 np.nan,
 np.nan,
 np.nan,
 np.nan,
 np.nan,
 np.nan,
 np.nan,
 np.nan]

KeyError: "['mac2'] not in index"

In [31]:
import pandas as pd

df = pd.read_csv('windows_dac_EPS1_3600_10.csv', index_col=0)

index_names = ['idxwindow', 'idxdaywindow', 'day', 'dac_family', 'mac']
cols_values=['qr', 'num DAC_RANK=1', 'num DAC_RANK=3', 'q', 'nx', 'qr_p1', 'q_p1', 'nx_p1', 'mac2', 'num DN', 'avg DN/QR', 'avg DN/Q', 'avg DN/NX', 'std DN/QR', 'std DN/Q', 'std DN/NX', 'max DN/QR', 'max DN/Q', 'max DN/NX' ]
df = df.set_index(index_names)

notm = 0
data = [[], []]
for idxday in range(10):
    for idxhour in range(24):
        for idxmw, mw in enumerate(dac_families):
            for idxmac, mac in enumerate(macs):
                idxwindow = idxhour + idxday * 24
                try:
                    df.loc[(idxwindow, idxhour, idxday, mw, mac)]
                    notm+=1
                except KeyError as e:
                    # pd.DataFrame()
                    # missings.append((idxwindow, idxhour, idxday, mw, mac), emptyrow)
                    data[0].append((idxwindow, idxhour, idxday, mw, mac))
                    data[1].append(emptyrow)
                    # df.at[(idxwindow, idxhour, idxday, mw, mac)] = emptyrow
                    pass
                pass
            pass
        pass
    pass

filler = pd.DataFrame(data[1], index=pd.MultiIndex.from_tuples(data[0], names=index_names), columns=cols_values)

df = pd.concat([df, filler]).sort_index().reset_index()

In [32]:
index_names = ['idxwindow', 'idxdaywindow', 'day', 'dac_family', 'mac']
 #, 'avg DN/QR', 'avg DN/Q', 'avg DN/NX', 'std DN/QR', 'std DN/Q', 'std DN/NX', 'max DN/QR', 'max DN/Q', 'max DN/NX' ]
# cols_values=['nx' ] #, 'avg DN/QR', 'avg DN/Q', 'avg DN/NX', 'std DN/QR', 'std DN/Q', 'std DN/NX', 'max DN/QR', 'max DN/Q', 'max DN/NX' ]

pairs = {}
labels_summary = []
for mac in macs:
    pairs[mac] = {}
    for dac_family in dac_families:
        train = df[(df.mac == mac) & (df.day < 5) & (df.dac_family == 'healthy')]
        test = df[(df.mac == mac) & (df.day >= 5) & (df.dac_family == dac_family)]

        train_label = df[(df.mac == mac) & (df.day < 5) & (df.dac_family == 'healthy')]['num DAC_RANK=1'] > 0
        test_label = df[(df.mac == mac) & (df.day >= 5) & (df.dac_family == dac_family)]['num DAC_RANK=1'] > 0

        pairs[mac][dac_family] = (train, test, train_label, test_label)
        labels_summary.append([mac, dac_family, train_label.sum(), test_label.sum()])
        pass
    pass

labels_summary = pd.DataFrame(labels_summary, columns=['mac', 'mw', 'train', 'test']).set_index(['mac', 'mw'])

In [37]:
from sklearn.ensemble import IsolationForest
import numpy as np

cols_values=['qr', 'nx', 'qr_p1', 'nx_p1' ]
for mac in macs:
    for dac_family in dac_families:

        train = pairs[mac][dac_family][0]
        test  = pairs[mac][dac_family][1]
        anomalies = (test['num DAC_RANK=1'] > 0).sum()

        if anomalies == 0:
            continue

        print(mac, dac_family, test['qr'].sum(), end=':')

        model = IsolationForest(contamination=0.2, random_state=90)
        model.fit(train[cols_values])

        predictions = model.predict(test[cols_values])

        print(f'\t\tanormals: {((predictions == -1) & (test['num DAC_RANK=1'] > 0)).sum()} / {anomalies}')

        pass


74:8e:f8:fb:80:7e virut 53401802:		anormals: 0 / 9
74:8e:f8:fb:80:7e modpack 53402261:		anormals: 27 / 114
74:8e:f8:fb:80:7e necurs 54416008:		anormals: 86 / 107
74:8e:f8:fb:80:7e pitou 53432848:		anormals: 9 / 33
00:e0:20:11:08:e6 virut 59957346:		anormals: 0 / 9
00:e0:20:11:08:e6 modpack 59958787:		anormals: 29 / 115
00:e0:20:11:08:e6 necurs 61174931:		anormals: 99 / 115
00:e0:20:11:08:e6 pitou 59981115:		anormals: 8 / 44
a6:3f:85:2f:c1:e0 virut 31548518:		anormals: 5 / 38
a6:3f:85:2f:c1:e0 modpack 31546569:		anormals: 30 / 115
a6:3f:85:2f:c1:e0 necurs 35676070:		anormals: 101 / 115
a6:3f:85:2f:c1:e0 pitou 31622556:		anormals: 10 / 44
00:04:96:41:28:00 virut 32422665:		anormals: 0 / 1
00:04:96:41:28:00 modpack 32423963:		anormals: 10 / 108
00:04:96:41:28:00 necurs 32599199:		anormals: 47 / 78
00:04:96:41:28:00 pitou 32432650:		anormals: 2 / 17
00:26:cb:32:f2:3f modpack 932429:		anormals: 4 / 17
00:26:cb:32:f2:3f necurs 949916:		anormals: 6 / 6


In [99]:
%pip install tensorflow

import numpy as np
from keras.models import Sequential
from keras.layers import Dense
from sklearn.preprocessing import StandardScaler

# Creiamo alcuni dati di esempio (traffico normale)
normal_data = np.random.normal(0, 1, (1000, 5))  # Traffico "normale"
anomalous_data = np.random.normal(10, 1, (200, 5))  # Traffico anomalo
test_data = np.vstack([normal_data, anomalous_data])

# Normalizza i dati
scaler = StandardScaler()
normal_data_scaled = scaler.fit_transform(normal_data)
test_data_scaled = scaler.transform(test_data)

# Creiamo un modello di autoencoder
autoencoder = Sequential()
autoencoder.add(Dense(8, activation='relu', input_dim=5))  # Encoder
autoencoder.add(Dense(5, activation='relu'))  # Bottleneck layer
autoencoder.add(Dense(8, activation='relu'))  # Decoder
autoencoder.add(Dense(5, activation='sigmoid'))  # Output layer

autoencoder.compile(optimizer='adam', loss='mean_squared_error')

# Alleniamo l'autoencoder sui dati normali
autoencoder.fit(normal_data_scaled, normal_data_scaled, epochs=50, batch_size=32, verbose=1)

# Calcoliamo l'errore di ricostruzione per i dati di test
reconstruction_error = autoencoder.evaluate(test_data_scaled, test_data_scaled)

print(f"Errore di ricostruzione sui dati di test: {reconstruction_error}")

# Una soglia di errore può essere impostata per decidere se un dato è anomalo
threshold = 0.1
predictions = (reconstruction_error > threshold).astype(int)

# Visualizza le anomalie (1: anomalo, 0: normale)
print("Anomalie rilevate:")
print(predictions)


  Using cached astunparse-1.6.3-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached flatbuffers-24.3.25-py2.py3-none-any.whl.metadata (850 bytes)
  Using cached gast-0.6.0-py3-none-any.whl.metadata (1.3 kB)
  Using cached google_pasta-0.2.0-py3-none-any.whl.metadata (814 bytes)
  Using cached libclang-18.1.1-1-py2.py3-none-macosx_11_0_arm64.whl.metadata (5.2 kB)
  Using cached opt_einsum-3.4.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached termcolor-2.5.0-py3-none-any.whl.metadata (6.1 kB)
  Using cached wrapt-1.16.0-cp312-cp312-macosx_11_0_arm64.whl.metadata (6.6 kB)
  Using cached ml_dtypes-0.4.1-cp312-cp312-macosx_10_9_universal2.whl.metadata (20 kB)
  Using cached Markdown-3.7-py3-none-any.whl.metadata (7.0 kB)
  Using cached tensorboard_data_server-0.7.2-py3-none-any.whl.metadata (1.1 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 239.6/239.6 MB 5.5 MB/s eta 0:00:0000:0100:01
Using cached astunparse-1.6.3-py2.py3-none-any.whl (12 kB)
Using cached flatbuffers-24.3.25-py2.py3-

ModuleNotFoundError: No module named 'distutils'

In [ ]:
import numpy as np
from keras.models import Sequential
from keras.layers import Dense
from sklearn.preprocessing import StandardScaler

# Creiamo alcuni dati di esempio (traffico normale)
normal_data = np.random.normal(0, 1, (1000, 5))  # Traffico "normale"
anomalous_data = np.random.normal(10, 1, (200, 5))  # Traffico anomalo
test_data = np.vstack([normal_data, anomalous_data])

# Normalizza i dati
scaler = StandardScaler()
normal_data_scaled = scaler.fit_transform(normal_data)
test_data_scaled = scaler.transform(test_data)

# Creiamo un modello di autoencoder
autoencoder = Sequential()
autoencoder.add(Dense(8, activation='relu', input_dim=5))  # Encoder
autoencoder.add(Dense(5, activation='relu'))  # Bottleneck layer
autoencoder.add(Dense(8, activation='relu'))  # Decoder
autoencoder.add(Dense(5, activation='sigmoid'))  # Output layer

autoencoder.compile(optimizer='adam', loss='mean_squared_error')

# Alleniamo l'autoencoder sui dati normali
autoencoder.fit(normal_data_scaled, normal_data_scaled, epochs=50, batch_size=32, verbose=1)

# Calcoliamo l'errore di ricostruzione per i dati di test
reconstruction_error = autoencoder.evaluate(test_data_scaled, test_data_scaled)

print(f"Errore di ricostruzione sui dati di test: {reconstruction_error}")

# Una soglia di errore può essere impostata per decidere se un dato è anomalo
threshold = 0.1
predictions = (reconstruction_error > threshold).astype(int)

# Visualizza le anomalie (1: anomalo, 0: normale)
print("Anomalie rilevate:")
print(predictions)


74:8e:f8:fb:80:7e virut 99171403:		anormals: 163 / 14
74:8e:f8:fb:80:7e modpack 99172766:		anormals: 163 / 199
74:8e:f8:fb:80:7e necurs 101375957:		anormals: 194 / 189
74:8e:f8:fb:80:7e pitou 99242148:		anormals: 163 / 73
74:8e:f8:fb:80:7e tofsee 99169632:		anormals: 163 / 1
00:e0:20:11:08:e6 virut 110081080:		anormals: 149 / 16
00:e0:20:11:08:e6 modpack 110084171:		anormals: 149 / 207
00:e0:20:11:08:e6 necurs 112586974:		anormals: 211 / 202
00:e0:20:11:08:e6 pitou 110124684:		anormals: 151 / 86
00:e0:20:11:08:e6 conficker 110091099:		anormals: 148 / 26
00:e0:20:11:08:e6 tofsee 110079477:		anormals: 149 / 1
a6:3f:85:2f:c1:e0 virut 61442648:		anormals: 113 / 72
a6:3f:85:2f:c1:e0 modpack 61439108:		anormals: 113 / 206
a6:3f:85:2f:c1:e0 necurs 70316212:		anormals: 207 / 202
a6:3f:85:2f:c1:e0 pitou 61579493:		anormals: 114 / 86
a6:3f:85:2f:c1:e0 conficker 61470257:		anormals: 113 / 26
a6:3f:85:2f:c1:e0 tofsee 61436958:		anormals: 113 / 1
00:04:96:41:28:00 virut 70045159:		anormals: 168 / 3